In [1]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="flan_t5_contextual_calibrated_yes_no_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [2]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [3]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)



---[ TableVault Record ]---
device: mps
---[ TableVault Record ]---



In [4]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
model.eval()

decoder_start_token_id = model.config.decoder_start_token_id
if decoder_start_token_id is None:
    decoder_start_token_id = tokenizer.pad_token_id

yes_ids = tokenizer.encode("yes", add_special_tokens=False)
no_ids = tokenizer.encode("no", add_special_tokens=False)

print("model_name:", model_name)
print("decoder_start_token_id:", decoder_start_token_id)
print("yes_ids:", yes_ids, "tokens:", tokenizer.convert_ids_to_tokens(yes_ids))
print("no_ids:", no_ids, "tokens:", tokenizer.convert_ids_to_tokens(no_ids))

assert len(yes_ids) == 1, f"'yes' must be a single token, got {yes_ids}"
assert len(no_ids) == 1, f"'no' must be a single token, got {no_ids}"

yes_id = yes_ids[0]
no_id = no_ids[0]



---[ TableVault Record ]---


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model_name: google/flan-t5-small
decoder_start_token_id: 0
yes_ids: [4273] tokens: ['▁yes']
no_ids: [150] tokens: ['▁no']
---[ TableVault Record ]---



In [5]:
prompt_template = (
    "Determine whether the following two sentences are paraphrases. "
    "Answer yes or no.\n"
    "Sentence 1: {sentence1}\n"
    "Sentence 2: {sentence2}\n"
    "Answer:"
)

def build_prompt(sentence1, sentence2):
    return prompt_template.format(sentence1=sentence1, sentence2=sentence2)

calibration_prompt = build_prompt("N/A", "N/A")
print(calibration_prompt)



---[ TableVault Record ]---
Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: N/A
Sentence 2: N/A
Answer:
---[ TableVault Record ]---



In [6]:
cal_enc = tokenizer(
    [calibration_prompt],
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt",
)
cal_enc = {k: v.to(device) for k, v in cal_enc.items()}
cal_decoder_input_ids = torch.full(
    (1, 1),
    decoder_start_token_id,
    dtype=torch.long,
    device=device,
)

with torch.no_grad():
    cal_logits = model(**cal_enc, decoder_input_ids=cal_decoder_input_ids).logits

baseline_yes_logit = float(cal_logits[0, 0, yes_id].detach().cpu())
baseline_no_logit = float(cal_logits[0, 0, no_id].detach().cpu())

print({"baseline_yes_logit": baseline_yes_logit, "baseline_no_logit": baseline_no_logit})



---[ TableVault Record ]---
{'baseline_yes_logit': -1.7861056327819824, 'baseline_no_logit': -2.062917470932007}
---[ TableVault Record ]---



In [7]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
prompts = [build_prompt(s1, s2) for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(ds))
print("positive_rate:", y_true.mean())
print("first_prompt:\n", prompts[0])



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
first_prompt:
 Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Answer:
---[ TableVault Record ]---



In [8]:
batch_size = 32
raw_yes_logits = []
raw_no_logits = []
cal_yes_logits = []
cal_no_logits = []
preds = []

with torch.no_grad():
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        batch_decoder_input_ids = torch.full(
            (len(batch_prompts), 1),
            decoder_start_token_id,
            dtype=torch.long,
            device=device,
        )

        logits = model(**enc, decoder_input_ids=batch_decoder_input_ids).logits[:, 0, :]

        batch_yes = logits[:, yes_id].detach().cpu().numpy()
        batch_no = logits[:, no_id].detach().cpu().numpy()

        batch_cal_yes = batch_yes - baseline_yes_logit
        batch_cal_no = batch_no - baseline_no_logit
        batch_preds = (batch_cal_yes > batch_cal_no).astype(np.int64)

        raw_yes_logits.extend(batch_yes.tolist())
        raw_no_logits.extend(batch_no.tolist())
        cal_yes_logits.extend(batch_cal_yes.tolist())
        cal_no_logits.extend(batch_cal_no.tolist())
        preds.extend(batch_preds.tolist())

y_pred = np.array(preds)
raw_yes_logits = np.array(raw_yes_logits)
raw_no_logits = np.array(raw_no_logits)
cal_yes_logits = np.array(cal_yes_logits)
cal_no_logits = np.array(cal_no_logits)

print("done")



---[ TableVault Record ]---


  0%|          | 0/13 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [9]:

vault.create_record_list("flan_prediction_logit_calibrated", column_names=["prediction","raw_yes_logits", "raw_no_logits", "cal_yes_logits", "cal_no_logits"])

for i in range(len(y_pred)):
    vault.append_record("flan_prediction_logit_calibrated", 
                        {
                            "prediction": int(y_pred[i]),
                            "raw_yes_logits": float(raw_yes_logits[i]),
                            "raw_no_logits": float(raw_no_logits[i]),
                            "cal_yes_logits": float(cal_yes_logits[i]),
                            "cal_no_logits": float(cal_no_logits[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "Per-example prediction dataset for the GLUE MRPC validation set produced by prompting google/flan-t5-small with a yes/no paraphrase question and applying contextual calibration. Each record corresponds to one input sentence pair from glue_mrpc_validation and stores the model\u2019s binary prediction (1 = paraphrase/yes, 0 = not paraphrase/no), the raw first-step logits for the yes and no tokens (raw_yes_logits, raw_no_logits), and the calibrated logits after subtracting baseline logits from a null calibration prompt (cal_yes_logits, cal_no_logits). This record list is the main intermediate output of the workflow: it preserves calibrated model scores for each example, supports error analysis and inspection, and is used to compute the downstream summary metrics in flan_t5_contextual_calibrated_yes_no_mrpc_summary."
embedding = get_embeddings(description)
vault.create_description("flan_prediction_logit_calibrated", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model prediction logits", "source": "glue/mrpc", "source_split": "validation", "size": "408", "model": "google/flan-t5-small", "prompt_type": "yes/no instruction prompt", "calibration": "contextual logit calibration", "label_space": "yes/no", "prediction_target": "paraphrase label", "input_fields": "sentence1,sentence2", "output_fields": "prediction,raw_yes_logits,raw_no_logits,cal_yes_logits,cal_no_logits"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_prediction_logit_calibrated", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [10]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.40441176470588236, 'f1': 0.24299065420560748}
                precision    recall  f1-score   support

not_paraphrase       0.34      0.98      0.51       129
    paraphrase       0.93      0.14      0.24       279

      accuracy                           0.40       408
     macro avg       0.64      0.56      0.38       408
  weighted avg       0.74      0.40      0.33       408

---[ TableVault Record ]---



In [11]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print(prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("raw_yes_logit:", float(raw_yes_logits[i]))
    print("raw_no_logit:", float(raw_no_logits[i]))
    print("calibrated_yes_logit:", float(cal_yes_logits[i]))
    print("calibrated_no_logit:", float(cal_no_logits[i]))



---[ TableVault Record ]---
idx: 0
Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Answer:
true: 1 pred: 0
raw_yes_logit: -2.405442714691162
raw_no_logit: -2.259373903274536
calibrated_yes_logit: -0.6193370819091797
calibrated_no_logit: -0.1964564323425293
idx: 1
Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
Sentence 2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
Answer:
true: 0 pred: 0
raw_yes_logit: -3.4676146507263184
raw_no_logit: -2.4009058475494385
calibrated_yes_logit: -1.681509017944336
calibrated_no_logit: -0.337988376617

In [12]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print(prompts[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("raw_yes_logit:", float(raw_yes_logits[i]))
    print("raw_no_logit:", float(raw_no_logits[i]))
    print("calibrated_yes_logit:", float(cal_yes_logits[i]))
    print("calibrated_no_logit:", float(cal_no_logits[i]))


vault.create_record_list("flan_t5_contextual_calibrated_yes_no_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("flan_t5_contextual_calibrated_yes_no_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "flan_prediction_logit_calibrated": [0, len(ds)]
                    })

summary

description = "Aggregate evaluation summary for the flan-t5-small contextual yes/no paraphrase classifier on the GLUE MRPC validation set after logit calibration. This dataset is a record list with one summary record containing the fields: accuracy (float), f1 (float), and classification_report (string with per-class precision/recall/F1 and support for not_paraphrase and paraphrase). It is produced after generating calibrated predictions in flan_prediction_logit_calibrated and compares those predictions against the ground-truth labels in glue_mrpc_validation. Its role in this workflow is to store run-level performance metrics for the full validation set, providing a compact summary of model quality for this notebook execution."
embedding = get_embeddings(description)
vault.create_description("flan_t5_contextual_calibrated_yes_no_mrpc_summary", description, embedding)

properties = {"artifact_type": "evaluation summary", "task": "paraphrase detection", "benchmark": "GLUE", "source": "glue/mrpc", "input_dataset": "glue_mrpc_validation", "split": "validation", "size": "408", "model": "google/flan-t5-small", "prompt_format": "yes/no seq2seq", "calibration": "contextual logit calibration", "label_space": "binary", "metrics": "accuracy,f1,classification_report"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_contextual_calibrated_yes_no_mrpc_summary", cat, embedding, prop)





---[ TableVault Record ]---
num_errors: 243
idx: 0
Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
Answer:
true: 1 pred: 0
raw_yes_logit: -2.405442714691162
raw_no_logit: -2.259373903274536
calibrated_yes_logit: -0.6193370819091797
calibrated_no_logit: -0.1964564323425293
idx: 3
Determine whether the following two sentences are paraphrases. Answer yes or no.
Sentence 1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
Sentence 2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
Answer:
true: 1 pred: 0
raw_yes_logit: -2.5131375789642334
raw_no_logit: -2.2279982566833496
calibrated_yes_logit: -0.727031946182251
calibrated_no_logit: -0.16508078575134277
idx: 5
Determin

In [13]:
description = "This notebook runs a zero-shot paraphrase detection experiment on the GLUE MRPC validation set using google/flan-t5-small. It frames each sentence pair as an instruction prompt asking whether the two sentences are paraphrases and scores the first decoder-step logits for the single-token answers yes and no. To reduce prompt bias, it applies contextual calibration by subtracting baseline yes/no logits computed from a null prompt with placeholder inputs (\u201cN/A\u201d, \u201cN/A\u201d), then predicts the label from the higher calibrated logit. The workflow loads MRPC examples from TableVault, generates batch predictions, stores raw and calibrated logits plus final predictions back into TableVault, computes evaluation metrics including accuracy, F1, and a classification report, inspects sample predictions and errors, and saves summary records and semantic descriptions for the prediction outputs, summary table, and overall notebook process." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("flan_t5_contextual_calibrated_yes_no_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text classification", "method": "zero-shot seq2seq yes/no prompting with contextual calibration", "calibration": "contextual calibration using baseline logits from N/A prompt", "model": "google/flan-t5-small", "model_family": "FLAN-T5", "dataset": "glue/mrpc", "dataset_split": "validation", "input_format": "sentence pair prompt", "label_space": "yes/no", "prediction_signal": "first-decoder-step logits for yes and no tokens", "evaluation": "accuracy, f1-score, classification report", "frameworks": "PyTorch, Hugging Face Transformers, Datasets, scikit-learn", "tracking": "TableVault", "embedding_model": "text-embedding-3-large", "artifacts": "per-example calibrated logits and summary metrics", "process_name": "flan_t5_contextual_calibrated_yes_no_mrpc"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("flan_t5_contextual_calibrated_yes_no_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

